In [ ]:
#|default_exp diff_nbs

# Notebook Diff

> CLI tool for semantic diffs between notebook versions, ignoring cell outputs and metadata

In [ ]:
#|export
import json, sys, os, argparse, subprocess
from difflib import unified_diff

In [ ]:
#|export
def _strip_nb(nb):
    "Strip outputs, execution counts, and cell-level metadata from a notebook dict, returning clean source lines."
    lines = []
    for i, cell in enumerate(nb.get('cells', [])):
        cell_type = cell.get('cell_type', 'code')
        lines.append(f'# %% [cell {i}] ({cell_type})\n')
        source = cell.get('source', [])
        if isinstance(source, str):
            source = source.splitlines(True)
        for line in source:
            if not line.endswith('\n'):
                line += '\n'
            lines.append(line)
        lines.append('\n')
    return lines

In [ ]:
#|export
def _load_nb(path):
    "Load a notebook from a file path and return the parsed JSON dict."
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
#|export
def _load_nb_from_git(rev, path):
    "Load a notebook from a git revision using `git show REV:path`."
    try:
        result = subprocess.run(
            ['git', 'show', f'{rev}:{path}'],
            capture_output=True, text=True, check=True
        )
        return json.loads(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error: could not retrieve '{path}' at revision '{rev}'.\n{e.stderr.strip()}", file=sys.stderr)
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"Error: '{path}' at revision '{rev}' is not valid JSON.\n{e}", file=sys.stderr)
        sys.exit(1)

In [ ]:
#|export
def diff_nbs(nb_a, nb_b, label_a='a', label_b='b', context=3):
    "Produce a unified diff of the source content between two notebook dicts."
    lines_a = _strip_nb(nb_a)
    lines_b = _strip_nb(nb_b)
    return list(unified_diff(lines_a, lines_b, fromfile=label_a, tofile=label_b, n=context))

In [ ]:
#|export
def main():
    "CLI entry point for fastai_diff_nbs."
    parser = argparse.ArgumentParser(
        prog='fastai_diff_nbs',
        description='Show semantic diffs between notebook versions, ignoring outputs and metadata.'
    )
    parser.add_argument('path', help='Path to the first (or only) notebook')
    parser.add_argument('path2', nargs='?', default=None, help='Path to the second notebook (for direct comparison)')
    parser.add_argument('--rev', default=None, help='Git revision to compare against (used with a single path)')
    parser.add_argument('--context', type=int, default=3, help='Number of context lines in the diff (default: 3)')
    args = parser.parse_args()

    if args.path2 and args.rev:
        parser.error('Cannot use --rev together with two positional paths.')

    if args.path2:
        # Direct comparison of two files
        nb_a = _load_nb(args.path)
        nb_b = _load_nb(args.path2)
        label_a, label_b = args.path, args.path2
    elif args.rev:
        # Compare working copy against a git revision
        nb_a = _load_nb_from_git(args.rev, args.path)
        nb_b = _load_nb(args.path)
        label_a = f'{args.rev}:{args.path}'
        label_b = args.path
    else:
        # Default: compare against HEAD
        nb_a = _load_nb_from_git('HEAD', args.path)
        nb_b = _load_nb(args.path)
        label_a = f'HEAD:{args.path}'
        label_b = args.path

    result = diff_nbs(nb_a, nb_b, label_a=label_a, label_b=label_b, context=args.context)
    if result:
        sys.stdout.write(''.join(result))
        sys.exit(1)
    else:
        sys.exit(0)

In [ ]:
#|export
if __name__ == '__main__':
    main()

In [ ]:
#|hide
# Test _strip_nb
test_nb = {
    'cells': [
        {'cell_type': 'code', 'source': ['print("hello")'], 'outputs': [{'text': 'hello'}], 'execution_count': 1, 'metadata': {'scrolled': True}},
        {'cell_type': 'markdown', 'source': ['# Title\n', 'Some text'], 'metadata': {}}
    ]
}
stripped = _strip_nb(test_nb)
assert '# %% [cell 0] (code)\n' in stripped
assert '# %% [cell 1] (markdown)\n' in stripped
assert 'print("hello")\n' in stripped
assert '# Title\n' in stripped
# Outputs should NOT appear
assert 'hello' not in ''.join(stripped).replace('print("hello")', '').replace('# Title', '')

In [ ]:
#|hide
# Test diff_nbs with identical notebooks
assert diff_nbs(test_nb, test_nb) == []

# Test diff_nbs with different notebooks
test_nb2 = {
    'cells': [
        {'cell_type': 'code', 'source': ['print("world")'], 'outputs': [], 'execution_count': None, 'metadata': {}},
        {'cell_type': 'markdown', 'source': ['# Title\n', 'Some text'], 'metadata': {}}
    ]
}
d = diff_nbs(test_nb, test_nb2)
assert len(d) > 0
assert any('hello' in line for line in d)
assert any('world' in line for line in d)

## Export -

In [ ]:
#|hide
from nbdev import nbdev_export
nbdev_export()